# ANLP Assignment 1 — Keys, W&B, Hugging Face & Kaggle

Everything to do with **credentials and publishing**, kept deliberately separate from
`01_train_local_rtx3050.ipynb` so that no secret ever ends up pasted into the training
notebook (and therefore into your submitted ZIP).

| Service | Why the assignment needs it | Required? |
|---|---|---|
| **Weights & Biases** | "Use W&B to log your training runs" | ✅ required |
| **Hugging Face** | "Host your pre-trained model checkpoints on HF" | ✅ required |
| **Kaggle** | Extra mirror for the checkpoints | optional |

> **🔒 Rule #1** — `.env`, `kaggle.json` and `_netrc` must **never** go into the ZIP you
> upload to Moodle. `.gitignore` already excludes them, and `make_submission.py`
> filters them out. Step 8 verifies this.

---
## Step 1 — Get your three keys

### 1a. Weights & Biases

1. Sign in at **<https://wandb.ai>** (use "Sign in with Google" if that is your IIIT account).
2. Go straight to **<https://wandb.ai/authorize>**.
3. Copy the key shown. It looks like either
   * a 40-character hex string (`0123abcd…`, older accounts), or
   * `wandb_v1_…` (~86 characters, newer accounts). **Both work.**

### 1b. Hugging Face

1. Sign in at **<https://huggingface.co>**.
2. Go to **<https://huggingface.co/settings/tokens>**.
3. **New token** → give it a name (e.g. `anlp-a1`).
4. ⚠️ **Permissions matter.** Pick either:
   * **Token type: Write** (simplest — full write access), or
   * **Fine-grained**, and then explicitly tick
     * ☑ *Create repositories*
     * ☑ *Write access to contents/settings of all repos you own*

   A fine-grained token **without** those two boxes will authenticate fine and then
   fail with `403 Forbidden` at push time. Step 5 checks this for you.
5. Copy the token — it starts with `hf_` and is ~37 characters.

### 1c. Kaggle *(optional)*

1. Sign in at **<https://www.kaggle.com>**.
2. Go to **<https://www.kaggle.com/settings>** → section **API** → **Create New Token**.
3. Your browser downloads **`kaggle.json`**, which contains exactly:
   ```json
   {"username": "yourname", "key": "0123456789abcdef0123456789abcdef"}
   ```

---
## Step 2 — Where to put the keys

Pick **Option A**. The other two are documented because tools sometimes look for them.

### ✅ Option A (recommended) — a `.env` file in the project root

Create a plain-text file named exactly **`.env`** next to `run_ablation.py`:

```
c:\Users\vidva\...\Assignment1\.env
```

with this content (no quotes, no spaces around `=`):

```ini
WANDB_API_KEY=wandb_v1_xxxxxxxxxxxxxxxxxxxxxxxx
WANDB_ENTITY=
WANDB_PROJECT=anlp-a1-blt-ablation

HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxx
HF_USERNAME=your-hf-username

KAGGLE_USERNAME=your-kaggle-username
KAGGLE_KEY=0123456789abcdef0123456789abcdef

ROLL_NUMBER=2021XXXXXX
```

Leave `WANDB_ENTITY` blank to use your personal workspace.

> **Windows tip — Notepad will betray you.** It saves `.env` as `.env.txt`. Either use
> VS Code, or in the Save dialog set *Save as type* → **All Files** and quote the name
> as `".env"`. The cell in Step 2a writes the file correctly for you, so you can skip
> the manual route entirely.

### Option B — Windows environment variables (persist across reboots)

In **PowerShell**, once:

```powershell
setx WANDB_API_KEY  "wandb_v1_xxxx"
setx HF_TOKEN       "hf_xxxx"
setx KAGGLE_USERNAME "yourname"
setx KAGGLE_KEY     "0123...."
```

Then **close and reopen** the terminal / restart the Jupyter kernel — `setx` only
affects *new* processes.

### Option C — the locations each tool natively looks in

| Tool | Windows path | Created by |
|---|---|---|
| W&B | `%USERPROFILE%\_netrc` → `C:\Users\vidva\_netrc` | `wandb login` |
| Hugging Face | `%USERPROFILE%\.cache\huggingface\token` | `hf auth login` |
| Kaggle | `%USERPROFILE%\.kaggle\kaggle.json` | you move the downloaded file there |

(On Windows W&B uses `_netrc`, **not** `.netrc` — that trips people up.)

### 2a. Write `.env` safely, without typing keys into a notebook cell

`getpass` masks the input, so your key never appears in the notebook output — and
therefore never in the file you submit. Run this cell and paste each value at the prompt
(press Enter to skip / keep an existing value).

In [ ]:
import os, sys, getpass
from pathlib import Path

PROJECT = Path(r"c:\Users\vidva\OneDrive - International Institute of Information Technology\Desktop\Course_Work_4-1\ANLP\Assignments\Assignment1")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
ENV = PROJECT / ".env"

FIELDS = [
    ("WANDB_API_KEY",   True,  "W&B API key        (https://wandb.ai/authorize)"),
    ("WANDB_ENTITY",    False, "W&B entity/team    (blank = personal workspace)"),
    ("WANDB_PROJECT",   False, "W&B project name   (default anlp-a1-blt-ablation)"),
    ("HF_TOKEN",        True,  "HF token           (hf_...  needs WRITE access)"),
    ("HF_USERNAME",     False, "HF username"),
    ("KAGGLE_USERNAME", False, "Kaggle username    (optional)"),
    ("KAGGLE_KEY",      True,  "Kaggle API key     (optional)"),
    ("ROLL_NUMBER",     False, "Your roll number   (used for the ZIP name)"),
]

existing = {}
if ENV.exists():
    for line in ENV.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            existing[k.strip()] = v.strip()

vals = dict(existing)
for name, secret, hint in FIELDS:
    cur = existing.get(name, "")
    shown = (cur[:4] + "..." + cur[-4:]) if (secret and len(cur) > 8) else cur
    prompt = f"{name}  [{shown or 'empty'}]  {hint}\n> "
    entry = getpass.getpass(prompt) if secret else input(prompt)
    entry = entry.strip()
    if entry:
        vals[name] = entry

vals.setdefault("WANDB_PROJECT", "anlp-a1-blt-ablation")

ENV.write_text(
    "# Auto-generated - DO NOT COMMIT OR SUBMIT THIS FILE.\n\n"
    + "\n".join(f"{k}={vals.get(k, '')}" for k, _, _ in FIELDS) + "\n",
    encoding="utf-8")
try:
    os.chmod(ENV, 0o600)
except Exception:
    pass
print(f"\nwrote {ENV}")

### 2b. Already have the keys in a text file?

`setup_keys.py` scrapes a free-form text file (any layout — bare keys on their own
lines, or `label: value` pairs), writes `.env`, and verifies all three services.
It recognises `hf_…`, `wandb_v1_…`, legacy 40-hex W&B keys and 32-hex Kaggle keys.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "setup_keys.py"],
                     cwd=str(PROJECT), capture_output=True, text=True).stdout)

---
## Step 3 — Load and verify

Nothing below prints a secret; keys are always shown masked.

In [ ]:
def load_env(path=None):
    path = Path(path or ENV)
    out = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                if v.strip():
                    out[k.strip()] = v.strip()
                    os.environ[k.strip()] = v.strip()
    return out

def mask(v):
    return "(empty)" if not v else (f"{v[:4]}...{v[-4:]} (len {len(v)})" if len(v) > 8 else "***")

E = load_env()
SECRET = {"WANDB_API_KEY", "HF_TOKEN", "KAGGLE_KEY"}
for k in ["WANDB_API_KEY","WANDB_ENTITY","WANDB_PROJECT","HF_TOKEN",
          "HF_USERNAME","KAGGLE_USERNAME","KAGGLE_KEY","ROLL_NUMBER"]:
    v = os.environ.get(k, "")
    print(f"  {k:16s} = {mask(v) if k in SECRET else (v or '(empty)')}")

In [ ]:
# --- W&B ---
try:
    import wandb
    ok = wandb.login(key=os.environ.get("WANDB_API_KEY") or None, relogin=True, verify=True)
    v = wandb.Api().viewer
    print(f"OK   W&B      : {v.username}   (default entity: {v.entity})")
    WANDB_ENTITY = os.environ.get("WANDB_ENTITY") or v.entity
except Exception as e:
    WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
    print(f"FAIL W&B      : {type(e).__name__}: {e}")

# --- Hugging Face ---
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    me = api.whoami()
    HF_USERNAME = me["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    auth = (me.get("auth") or {}).get("accessToken") or {}
    print(f"OK   HF       : {HF_USERNAME}   (token role: {auth.get('role','?')})")
    if auth.get("role") == "fineGrained":
        print("     NOTE: fine-grained token - Step 5 will confirm it can actually write.")
except Exception as e:
    HF_USERNAME = os.environ.get("HF_USERNAME", "")
    print(f"FAIL HF       : {type(e).__name__}: {e}")

# --- Kaggle ---
try:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        from kaggle.api.kaggle_api_extended import KaggleApi
        KaggleApi().authenticate()
        print(f"OK   Kaggle   : {os.environ['KAGGLE_USERNAME']}")
    else:
        print("SKIP Kaggle   : KAGGLE_USERNAME / KAGGLE_KEY not set (optional)")
except Exception as e:
    print(f"FAIL Kaggle   : {type(e).__name__}: {e}")

---
## Step 4 — Weights & Biases: collect the run links

Training (notebook 01) already streamed per-step loss, per-epoch validation metrics,
LR, throughput, peak GPU memory and the final test metrics to W&B. This step just
gathers the URLs the report has to cite.

In [ ]:
import json, wandb
PROJECT_NAME = os.environ.get("WANDB_PROJECT", "anlp-a1-blt-ablation")
api = wandb.Api()

rows = []
try:
    runs = api.runs(f"{WANDB_ENTITY}/{PROJECT_NAME}")
    for r in runs:
        rows.append({
            "config": r.config.get("config", r.name),
            "name": r.name, "state": r.state, "url": r.url,
            "bit_acc": r.summary.get("test/bit_accuracy"),
            "seq_acc": r.summary.get("test/sequence_accuracy"),
        })
except Exception as e:
    print("could not list runs:", e)

print(f"W&B project: https://wandb.ai/{WANDB_ENTITY}/{PROJECT_NAME}\n")
for r in sorted(rows, key=lambda x: str(x["config"])):
    print(f"  {str(r['config']):<4} {r['state']:<9} {r['url']}")

(PROJECT / "outputs" / "wandb_runs.json").write_text(
    json.dumps({"project_url": f"https://wandb.ai/{WANDB_ENTITY}/{PROJECT_NAME}",
                "runs": rows}, indent=2), encoding="utf-8")
print(f"\nsaved -> outputs/wandb_runs.json")

**Offline runs?** If you trained with `--wandb offline` (e.g. no internet at the time),
upload them now:

```python
import glob, subprocess, sys
for d in sorted(glob.glob(str(PROJECT / "wandb" / "offline-run-*"))):
    subprocess.run([sys.executable, "-m", "wandb", "sync", d])
```

---
## Step 5 — Push the checkpoints to Hugging Face

Creates one model repo holding all five configurations, each in its own folder with
`best.pt`, `config.json`, `metrics.json`, `history.json`, the tokenizers and the
training-curve plot — plus an auto-generated model card at the repo root.

In [ ]:
HF_REPO = f"{HF_USERNAME}/anlp-a1-transformer-blt-ablation"
print("target repo:", HF_REPO)

# Confirm the token really has write permission BEFORE uploading ~30 MB.
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
try:
    api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True, private=False)
    print("OK  write permission confirmed (repo created or already exists)")
except Exception as e:
    print(f"FAIL {type(e).__name__}: {e}")
    print("\n-> Your token cannot create repos. Go to "
          "https://huggingface.co/settings/tokens ,\n"
          "   create a token with type 'Write' (or tick 'Create repositories' +\n"
          "   'Write access to contents/settings of all repos'), update .env, rerun Step 3.")

In [ ]:
rc = subprocess.run(
    [sys.executable, "-m", "src.push_to_hf", "--repo-id", HF_REPO, "--out-dir", "outputs"],
    cwd=str(PROJECT))
print("exit", rc.returncode)
print("\nModel page:", f"https://huggingface.co/{HF_REPO}")

---
## Step 6 — Push the checkpoints to Kaggle

`src/push_to_kaggle.py` publishes to **both** Kaggle surfaces:

* **Kaggle Models** → `https://www.kaggle.com/models/<you>/anlp-a1-transformer-blt-ablation`
* **Kaggle Datasets** → `https://www.kaggle.com/datasets/<you>/anlp-a1-transformer-blt-ablation`

Datasets are the easier thing to attach to a Kaggle notebook; Models is the "proper"
home for weights. Skip this whole section if you did not set up Kaggle keys.

In [ ]:
# Dry run first: stage the files and show what would be uploaded.
subprocess.run([sys.executable, "-m", "src.push_to_kaggle", "--dry-run"], cwd=str(PROJECT))

In [ ]:
# The real upload. Add --public to make the dataset publicly visible.
subprocess.run([sys.executable, "-m", "src.push_to_kaggle",
                "--target", "both", "--public"], cwd=str(PROJECT))

**If Kaggle rejects the slug**, it is already taken by another of your datasets — pass
a different one:

```python
subprocess.run([sys.executable, "-m", "src.push_to_kaggle",
                "--slug", "anlp-a1-blt-ablation-v2", "--target", "both"], cwd=str(PROJECT))
```

---
## Step 7 — Collect every link for the report and README

In [ ]:
import json
links = {
    "wandb_project": f"https://wandb.ai/{WANDB_ENTITY}/{PROJECT_NAME}",
    "huggingface":   f"https://huggingface.co/{HF_REPO}",
    "kaggle_model":  f"https://www.kaggle.com/models/{os.environ.get('KAGGLE_USERNAME','')}/anlp-a1-transformer-blt-ablation",
    "kaggle_dataset":f"https://www.kaggle.com/datasets/{os.environ.get('KAGGLE_USERNAME','')}/anlp-a1-transformer-blt-ablation",
}
wr = PROJECT / "outputs" / "wandb_runs.json"
if wr.exists():
    links["wandb_runs"] = {r["config"]: r["url"] for r in json.loads(wr.read_text())["runs"]}

(PROJECT / "outputs" / "links.json").write_text(json.dumps(links, indent=2), encoding="utf-8")
print(json.dumps(links, indent=2))
print("\nPaste these into README.md and Section 'Links' of the report.")

---
## Step 8 — Build the submission ZIP

`make_submission.py` assembles `<rollnumber>_assignment1/` in exactly the layout the
brief specifies, copies `Report.pdf` in, and **excludes every secret** (`.env`,
`kaggle.json`, `_netrc`, `wandb/`, `__pycache__/`).

In [ ]:
ROLL = os.environ.get("ROLL_NUMBER") or input("Roll number: ").strip()
subprocess.run([sys.executable, "make_submission.py", "--rollnumber", ROLL], cwd=str(PROJECT))

In [ ]:
# Safety net: prove no secret made it into the ZIP.
import zipfile, re
zp = PROJECT / f"{ROLL}_assignment1.zip"
if zp.exists():
    names = zipfile.ZipFile(zp).namelist()
    print(f"{zp.name}: {len(names)} entries, "
          f"{zp.stat().st_size/1024**2:.1f} MB\n")
    bad = [n for n in names
           if re.search(r"(^|/)\.env$|kaggle\.json|_netrc|\.git/|__pycache__|/wandb/", n)]
    print("SECRETS FOUND:", bad if bad else "none  ✅")
    print("\nTop-level entries:")
    for n in sorted({n.split('/')[1] for n in names if n.count('/') >= 1})[:20]:
        print("   ", n)
else:
    print("ZIP not found - check the output above.")

---
## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `wandb: Network error` / hangs at start | corporate/campus firewall | `set WANDB_MODE=offline`, train, then `wandb sync wandb/offline-run-*` |
| W&B logs to the wrong team | default entity is a team | set `WANDB_ENTITY=` to your **personal** username in `.env` |
| HF `403 Forbidden` on push | fine-grained token lacking write scope | new token with type **Write** (Step 1b) |
| HF `401 Unauthorized` | token not loaded | re-run Step 3; check `.env` is not saved as `.env.txt` |
| `OSError: kaggle.json not found` | Kaggle creds missing | put `kaggle.json` at `%USERPROFILE%\.kaggle\kaggle.json`, or set the two env vars |
| Kaggle `409 Conflict` | slug already used | pass a different `--slug` |
| `.env` values ignored | Notepad saved `.env.txt` | show file extensions in Explorer and rename |
| Keys leaked into a notebook output | you `print()`ed them | *Cell → All Output → Clear*, save, and **rotate the key** |

### If a key is ever exposed, rotate it immediately
* W&B — <https://wandb.ai/settings> → *Danger zone* → reset API key
* HF — <https://huggingface.co/settings/tokens> → *Invalidate* → create a new one
* Kaggle — <https://www.kaggle.com/settings> → *Expire API Token* → *Create New Token*